# Outbound Nursing Team Call Analysis - Revised

This notebook revisits the original analysis after I noticed the data had a structural pattern that the first pass missed. The original notebook is preserved in this repository for comparison. This revision keeps the same business questions and the same data, but handles the screening completion records differently and adds two findings that fell out of that change.

The short version. The original analysis used `drop_duplicates(keep='first')`-style handling on screening records where the same patient had both a 1 and a 0 for completing the same screening on the same date. That kept whichever row happened to come first. Looking at the data again, 152 of 369 appointment groups have these contradictory completion flags, and no appointment group has only 0 records. The 0s are not real missed visits. They are data entry inconsistencies. The fix is to look at the date column instead. All screenings for a patient are scheduled on the same day, so one date for a screening type means it was completed at first visit, and multiple dates means it was rescheduled. That logic bypasses the contradictions and uses what the data is actually telling you.

## Setup

In [ ]:
library(tidyverse)
library(lubridate)
library(rstatix)
library(ggplot2)
library(scales)

raw <- read_csv("outbound_call_nursing_team.csv", show_col_types = FALSE)

# Full-row deduplication
df <- raw %>% distinct()

cat("Raw rows:", nrow(raw), "\n")
cat("After full-row dedup:", nrow(df), "rows,", n_distinct(df$patient_id), "patients\n")

## The contradictions issue

Restricting to records with a valid screening date and a primary screening type (BCS, CBP, COL, EED, OMW), I grouped by patient, screening type, and date. Each unique combination is an appointment group. The completion flag values within each group tell the story.

In [ ]:
primary_types <- c("BCS", "CBP", "COL", "EED", "OMW")

df_dated <- df %>%
  filter(!is.na(screening_date), screening_type %in% primary_types) %>%
  mutate(screening_date = ymd(screening_date))

# Look at completion flag values within each appointment group
appt_groups <- df_dated %>%
  filter(screening_completed_ind %in% c("1.0", "0.0")) %>%
  group_by(patient_id, screening_type, screening_date) %>%
  summarise(
    flag_set = paste(sort(unique(screening_completed_ind)), collapse = ","),
    .groups = "drop"
  )

appt_groups %>% count(flag_set, name = "n_groups")

Of the 369 appointment groups, 217 have only completed records and 152 have both a completed and a not-completed record on the same date. Zero appointment groups have only not-completed records. If the 0s reflected actual missed visits, you would expect to see at least some groups with only 0s. The pattern is consistent with data entry inconsistencies where the same scheduled visit was sometimes recorded twice with conflicting flags. There was no standard process for data entry, or at least not one that was followed consistently.

Trying to resolve which flag is correct is the wrong question. The right question is whether the screening was completed at the first scheduled visit, and the date column answers that directly.

## Date logic

All screenings for a patient are scheduled on the same day. One unique scheduled date for a screening type means the patient completed it at first visit. Multiple unique dates means the patient was rescheduled. I count unique screening dates per patient and screening type.

In [ ]:
appt <- df_dated %>%
  group_by(patient_id, screening_type) %>%
  summarise(n_dates = n_distinct(screening_date), .groups = "drop") %>%
  mutate(completed_first_visit = as.integer(n_dates == 1))

# Per-patient outcome: did they complete every one of their screenings at first visit?
patient_completion <- appt %>%
  group_by(patient_id) %>%
  summarise(
    n_types = n(),
    n_completed_first = sum(completed_first_visit),
    .groups = "drop"
  ) %>%
  mutate(all_first_visit = as.integer(n_completed_first == n_types))

cat("Patients in analysis:", nrow(patient_completion), "\n")
cat("Completed all at first visit:", sum(patient_completion$all_first_visit), "\n")
cat("Did not complete all at first visit:", sum(patient_completion$all_first_visit == 0), "\n")

132 patients are in the analysis (the remaining 34 of 166 have only missing values for `screening_completed_ind` and `screening_date` and cannot be classified). Under date logic, 80 patients completed every one of their screenings at first visit, and 52 did not. The original analysis classified 111 of these 132 patients as completed and only 21 as not completed because the contradiction handling defaulted toward the 1 records. The corrected split has much more variation in the outcome, which feeds directly into Q2.

## Q2 — Compliance by number of eligible screening types

In [ ]:
patient_completion %>%
  group_by(n_types) %>%
  summarise(
    n = n(),
    completed_all_first = sum(all_first_visit),
    rate = mean(all_first_visit),
    .groups = "drop"
  )

In [ ]:
# Logistic regression: probability of completing all screenings at first visit
# as a function of how many screening types the patient is eligible for
fit <- glm(all_first_visit ~ n_types,
           data = patient_completion,
           family = binomial)

summary(fit)

cat("\nOdds ratio per additional screening type:",
    round(exp(coef(fit)["n_types"]), 3), "\n")

Under date logic the Q2 result is highly significant (p < 1e-7), and the odds of completing all screenings at first visit drop by about 86% with each additional eligible screening type. The original analysis came out marginal at p = 0.058. Same 132 patients, same model, different outcome classification.

The qualitative recommendation from the original analysis still holds: the program should prioritize patients with multiple screenings. The strength of the evidence behind that recommendation is now much higher.

## Proactive call timing

A finding that fell out of working with the date column. For every called patient with a valid screening date, the latest call date comes before the first scheduled screening date. The program is proactive outreach, not follow-up after missed visits.

In [ ]:
# Build patient-level call status
patient_status <- df %>%
  mutate(reached_num = suppressWarnings(as.numeric(reached_ind))) %>%
  group_by(patient_id) %>%
  summarise(
    ever_called  = any(!is.na(latest_call_date)),
    ever_reached = any(reached_num == 1, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(status = case_when(
    !ever_called  ~ "Not Called",
    ever_reached  ~ "Reached",
    TRUE          ~ "Not Reached"
  ))

patient_status %>% count(status)

In [ ]:
# Gap between call and first scheduled visit
call_timing <- df %>%
  filter(!is.na(latest_call_date), !is.na(screening_date)) %>%
  mutate(
    latest_call_date = ymd(latest_call_date),
    screening_date   = ymd(screening_date)
  ) %>%
  group_by(patient_id) %>%
  summarise(
    first_screening = min(screening_date),
    latest_call     = first(latest_call_date),
    .groups = "drop"
  ) %>%
  mutate(gap_days = as.integer(first_screening - latest_call)) %>%
  inner_join(patient_status %>% select(patient_id, status), by = "patient_id")

cat("Called patients with valid dates:", nrow(call_timing), "\n")
cat("Calls before first scheduled visit:",
    sum(call_timing$gap_days > 0), "of", nrow(call_timing), "\n\n")

call_timing %>%
  group_by(status) %>%
  summarise(
    n = n(),
    mean_gap_days = round(mean(gap_days), 1),
    median_gap_days = median(gap_days),
    .groups = "drop"
  )

All 81 called patients with valid dates have their call before their first scheduled screening. The mean gap is about 66 days for Reached patients and 37 days for Not Reached patients. This reframes how to read everything else. The program is not chasing patients who already missed visits. It is calling patients in advance to nudge them toward completion.

## Stratified analysis by number of screening types

Q3 in the original analysis asked whether reached patients perform better than not reached patients. Treating call status as a single program-wide variable obscured a more useful pattern. Once you stratify by number of screening types, the call-status effect only shows up for single-type patients.

In [ ]:
# Reschedule counts per patient (total extra visits across all their screening types)
patient_reschedule <- appt %>%
  group_by(patient_id) %>%
  summarise(n_reschedules = sum(n_dates - 1), .groups = "drop")

merged <- patient_completion %>%
  inner_join(patient_reschedule, by = "patient_id") %>%
  inner_join(patient_status %>% select(patient_id, status), by = "patient_id")

# Single-type patients
single <- merged %>% filter(n_types == 1)
cat("Single-type patients:", nrow(single), "\n\n")

single %>%
  group_by(status) %>%
  summarise(
    n = n(),
    mean_reschedules = round(mean(n_reschedules), 3),
    .groups = "drop"
  )

kruskal.test(n_reschedules ~ status, data = single)

In [ ]:
# Multi-type patients (2 or more screening types)
multi <- merged %>% filter(n_types >= 2)
cat("Multi-type patients:", nrow(multi), "\n\n")

multi %>%
  group_by(status) %>%
  summarise(
    n = n(),
    mean_reschedules = round(mean(n_reschedules), 3),
    .groups = "drop"
  )

kruskal.test(n_reschedules ~ status, data = multi)

Among single-type patients, call status does affect outcomes (Kruskal-Wallis p ≈ 0.013 on reschedule counts). Among patients with two or more screening types there is no detectable call-status effect. The original analysis would not have surfaced this because it treated call status as a single program-wide variable.

## Patient-level first visit completion by call status

For completeness, the patient-level first-visit completion rate by call status. These are per-patient and not directly comparable to the per-event completion percentages reported in the original analysis.

In [ ]:
merged %>%
  group_by(status) %>%
  summarise(
    n = n(),
    completed_all_first = sum(all_first_visit),
    rate = round(mean(all_first_visit), 3),
    .groups = "drop"
  )

Reached and Not Reached patients are not significantly different from each other at the patient level (48% vs 38%, Fisher p ≈ 0.46). The qualitative finding from the original analysis that reached and not reached patients perform similarly survives. Not Called patients show much higher first-visit completion (85%), but that is mostly driven by call status and number of screening types being correlated: Not Called patients in this dataset are concentrated in the single-screening group, which is the group with the highest first-visit completion overall.

## Summary

Three things changed in this revision.

First, the contradictory completion flags are now handled by reading dates instead of resolving 0/1 conflicts. That moves the Q2 result from marginal (p = 0.058) to highly significant (p < 1e-7) with the same model and the same patients.

Second, the call timing analysis shows the program is proactive outreach, not follow-up. All called patients have their call before their first scheduled screening. This changes how the program effectiveness question should be framed.

Third, stratifying by number of screening types shows the call-status effect is real for single-type patients and undetectable for patients with two or more types. The original analysis missed this by treating call status as a single program-wide variable.

The original qualitative recommendation, that the program should prioritize patients with multiple screenings, still holds and is now backed by a much stronger statistical result.